# 78 — Multi-Fingerprint Diversity Ensemble

Different fingerprints capture different aspects of molecular SAR:
- ECFP2: heavy atom environments (very local)
- ECFP4: standard circular (default)
- ECFP6: larger circular (more context)
- FCFP4: feature-based (pharmacophoric)
- MACCS: 166 predefined structural keys
- RDKit: path-based (bond topology)
- Topological torsion: 3D-proxy shape descriptor

Train LGBM on each → OOF predictions → ElasticNetCV meta-learner.
Diversity of view = better ensemble even with same model class.


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m

In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 149


In [4]:
from rdkit import Chem
from rdkit.Chem import MACCSkeys, AllChem, RDKFingerprint
from rdkit.Chem import rdMolDescriptors
from sklearn.linear_model import ElasticNetCV
import warnings

def smiles_to_fps(smiles_list):
    """Generate 7 fingerprint types for each SMILES. Returns dict of (N,bits) arrays."""
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    valid = [m is not None for m in mols]
    mols_v = [m for m in mols if m is not None]

    def fp_array(fn, n_bits):
        arr = np.zeros((len(smiles_list), n_bits), dtype=np.float32)
        idx = 0
        for i, ok in enumerate(valid):
            if ok:
                try:
                    bv = fn(mols_v[idx])
                    from rdkit.DataStructs import ConvertToNumpyArray
                    ConvertToNumpyArray(bv, arr[i])
                except: pass
                idx += 1
        return arr

    fps = {
        "ecfp2": fp_array(lambda m: AllChem.GetMorganFingerprintAsBitVect(m, 1, 2048), 2048),
        "ecfp4": fp_array(lambda m: AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048), 2048),
        "ecfp6": fp_array(lambda m: AllChem.GetMorganFingerprintAsBitVect(m, 3, 2048), 2048),
        "fcfp4": fp_array(lambda m: AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048, useFeatures=True), 2048),
        "maccs": fp_array(lambda m: MACCSkeys.GenMACCSKeys(m), 167),
        "rdkit6": fp_array(lambda m: RDKFingerprint(m, maxPath=6, fpSize=2048), 2048),
        "torsion": fp_array(lambda m: rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(m, 2048), 2048),
    }
    return fps

print("Computing 7 fingerprint types for training set...", flush=True)
fps_dict_tr = smiles_to_fps(tr["smiles"].tolist())
print("Computing 7 fingerprint types for test set...", flush=True)
fps_dict_te = smiles_to_fps(te["smiles"].tolist())
for k,v in fps_dict_tr.items():
    print(f"  {k}: {v.shape}")


Computing 7 fingerprint types for training set...


[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerator
[22:22:56] DEPRECATION WARNING: please use MorganGenerat

[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerat

[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerat

[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerator
[22:22:57] DEPRECATION WARNING: please use MorganGenerat

...

Computing 7 fingerprint types for test set...


[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerat

[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerator
[22:23:09] DEPRECATION WARNING: please use MorganGenerat

  ecfp2: (4139, 2048)
  ecfp4: (4139, 2048)
  ecfp6: (4139, 2048)
  fcfp4: (4139, 2048)
  maccs: (4139, 167)
  rdkit6: (4139, 2048)
  torsion: (4139, 2048)


[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23:10] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[22:23

In [5]:
# Train one LGBM per fingerprint type
print("\n=== Per-fingerprint scaffold CV ===", flush=True)
oof_per_fp = {}
te_per_fp  = {}
metrics_per_fp = {}

for fp_name, X_fp in fps_dict_tr.items():
    X_fp_te = fps_dict_te[fp_name]
    oof_fp = np.full(len(y_tr), np.nan)
    print(f"\n--- {fp_name} ({X_fp.shape[1]}d) ---", flush=True)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        m = lgb.train(LGBM, lgb.Dataset(X_fp[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_fp[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
        oof_fp[va_idx] = m.predict(X_fp[va_idx])
    oof_per_fp[fp_name] = oof_fp
    m_fp = lgb.train(LGBM, lgb.Dataset(X_fp, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
    te_per_fp[fp_name]  = m_fp.predict(X_fp_te)
    metrics_per_fp[fp_name] = full_metrics(y_tr, oof_fp, cliff_pairs, fp_name)

print("\n=== Summary per fingerprint ===")
print(pd.DataFrame(metrics_per_fp).T[["RAE","MAE","R2","Spearman","Cliff_acc" if "Cliff_acc" in metrics_per_fp.get("ecfp4",{}) else "RAE"]].round(4).to_string())



=== Per-fingerprint scaffold CV ===



--- ecfp2 (2048d) ---


  [ecfp2] RAE=0.6320 MAE=0.5750 R²=0.5303 r=0.7282 ρ=0.6795 τ=0.4907  Cliff=nan

--- ecfp4 (2048d) ---


  [ecfp4] RAE=0.6486 MAE=0.5901 R²=0.5184 r=0.7201 ρ=0.6618 τ=0.4748  Cliff=nan

--- ecfp6 (2048d) ---


  [ecfp6] RAE=0.6631 MAE=0.6033 R²=0.4954 r=0.7040 ρ=0.6440 τ=0.4592  Cliff=nan

--- fcfp4 (2048d) ---


  [fcfp4] RAE=0.6493 MAE=0.5908 R²=0.4969 r=0.7050 ρ=0.6567 τ=0.4732  Cliff=nan

--- maccs (167d) ---


  [maccs] RAE=0.6940 MAE=0.6314 R²=0.4334 r=0.6584 ρ=0.6068 τ=0.4321  Cliff=nan

--- rdkit6 (2048d) ---


  [rdkit6] RAE=0.6857 MAE=0.6239 R²=0.4582 r=0.6779 ρ=0.6234 τ=0.4427  Cliff=nan

--- torsion (2048d) ---


  [torsion] RAE=0.6812 MAE=0.6198 R²=0.4662 r=0.6828 ρ=0.6290 τ=0.4489  Cliff=nan

=== Summary per fingerprint ===
            RAE     MAE      R2  Spearman  Cliff_acc
ecfp2    0.6320  0.5750  0.5303    0.6795        NaN
ecfp4    0.6486  0.5901  0.5184    0.6618        NaN
ecfp6    0.6631  0.6033  0.4954    0.6440        NaN
fcfp4    0.6493  0.5908  0.4969    0.6567        NaN
maccs    0.6940  0.6314  0.4334    0.6068        NaN
rdkit6   0.6857  0.6239  0.4582    0.6234        NaN
torsion  0.6812  0.6198  0.4662    0.6290        NaN


In [6]:
# ElasticNetCV meta-learner on OOF stack
OOF_stack = np.column_stack([oof_per_fp[k] for k in fps_dict_tr])
TE_stack  = np.column_stack([te_per_fp[k]  for k in fps_dict_tr])
valid_rows = np.isfinite(OOF_stack).all(1)

from sklearn.linear_model import ElasticNetCV
meta = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=5000)
meta.fit(OOF_stack[valid_rows], y_tr[valid_rows])
print(f"\nMeta-learner weights: {dict(zip(fps_dict_tr.keys(), meta.coef_.round(3)))}")
print(f"Non-zero FPs: {(meta.coef_ != 0).sum()}/{len(fps_dict_tr)}")

oof = meta.predict(OOF_stack)
oof = np.clip(oof, y_tr.min()-0.5, y_tr.max()+0.5)
m_ens = full_metrics(y_tr, oof, cliff_pairs, "multi_fp_ensemble")
m_ens_a = full_metrics(y_tr[active_mask], oof[active_mask], "multi_fp_ensemble [active]")
print(pd.DataFrame([m_ens, m_ens_a], index=["overall","active"]).round(4).to_string())
te_preds = np.clip(meta.predict(TE_stack), y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED/"oof_per_fp_stack.npy", OOF_stack)
np.save(DATA_PROCESSED/"oof_multi_fp_ensemble.npy", oof)
np.save(DATA_PROCESSED/"te_oof_multi_fp_ensemble.npy", te_preds)
sub = pd.DataFrame({"Molecule Name":te["name"].values,"pEC50":te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"78_multi_fp_ensemble.csv"; sub.to_csv(p,index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")



Meta-learner weights: {'ecfp2': np.float64(0.255), 'ecfp4': np.float64(0.17), 'ecfp6': np.float64(0.042), 'fcfp4': np.float64(0.198), 'maccs': np.float64(0.154), 'rdkit6': np.float64(0.16), 'torsion': np.float64(0.205)}
Non-zero FPs: 7/7
  [multi_fp_ensemble] RAE=0.5897 MAE=0.5365 R²=0.5830 r=0.7636 ρ=0.7203 τ=0.5278  Cliff=nan
            RAE     MAE      R2  Pearson  Spearman  Kendall  Cliff_acc
overall  0.5897  0.5365  0.5830   0.7636    0.7203   0.5278        NaN
active   3.7342  0.7831 -9.6812   0.0578    0.0743   0.0502        NaN
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\78_multi_fp_ensemble.csv
Test: min=2.44 med=5.14 max=6.19
